# 임베딩 모델 4종 비교 (BGE-m3 / BGE-m3-ko / Qwen3-Embedding-8B / Nemotron-3-Embed-8B)

기존 `src/eval_retrieval.py`와 같은 방법론(testset_all.jsonl 169문항 + 꼬리 프로브 4문항, Recall@k/MRR/AnswerRecall@5)으로
**청킹 방식은 이미 검증된 `all`(FAQ+표 처치) 모드로 고정**하고, 임베딩 모델만 바꿔가며 비교한다.

GPU 필요 (Colab GPU 런타임 권장 — Runtime > Change runtime type > GPU). 8B 모델 2개는 bf16으로도
가중치만 ~16GB라 세션당 GPU 메모리 넉넉한 런타임(A100 등)이 아니면 모델 하나씩 순차 처리 후 반드시 언로드한다.

In [1]:
!pip install -q -U sentence-transformers transformers accelerate einops pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 141.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 164.5 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


## 0. (선택) Drive 마운트 — 임베딩 캐시 영구 저장용
Colab 세션이 끊기면 로컬 디스크는 초기화되지만 Drive는 유지된다. 8B 모델 재인코딩은 오래 걸리니 캐시를 Drive에 둔다.

In [2]:
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


## 1. 리포 경로 설정
Colab이면 리포를 클론해야 한다(비공개 리포면 토큰 필요). VS Code Remote-SSH로 이 리포가 이미 원격 머신에
있는 상태로 노트북을 여는 경우엔 클론 없이 바로 다음 셀로 넘어가면 된다.

In [3]:
from pathlib import Path
import sys

if IN_COLAB and not Path("/content/kdic-crawler").exists():
    # 필요 시 주소/브랜치 수정 (비공개 리포면 !git clone https://<TOKEN>@github.com/... 형태로)
    !git clone https://github.com/likelion-8/kdic-crawler.git /content/kdic-crawler

REPO_ROOT = Path("/content/kdic-crawler") if IN_COLAB else Path.cwd()
assert (REPO_ROOT / "src" / "chunking.py").exists(), f"src/chunking.py 못 찾음: {REPO_ROOT} — REPO_ROOT 직접 수정 필요"
sys.path.insert(0, str(REPO_ROOT / "src"))
print("REPO_ROOT:", REPO_ROOT)

Cloning into '/content/kdic-crawler'...
remote: Enumerating objects: 792, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 792 (delta 15), reused 19 (delta 12), pack-reused 747 (from 2)
Receiving objects: 100% (792/792), 5.27 MiB | 11.08 MiB/s, done.
Resolving deltas: 100% (505/505), done.
REPO_ROOT: /content/kdic-crawler


In [4]:
import gc
import hashlib
import json

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from chunking import build_units
from retrieval import PageRanked
from eval_retrieval import ROOT, load_testset, evaluate, answer_recall, by_type_mrr

assert torch.cuda.is_available(), "GPU 런타임으로 바꿔서 다시 실행 (Runtime > Change runtime type > GPU)"

## 2. 코퍼스 / 테스트셋 로드
청킹 방식은 `docs/retrieval_experiment_results.md`에서 이미 검증된 `all`(FAQ QA쌍 + 표 3행묶음) 모드로 고정.
여기서 모델별 성능 차이만 보는 게 목적이라 청킹까지 같이 바꾸면 변수가 뒤섞인다.

In [5]:
MODE = "all"
uids, texts, unit2page = build_units(MODE)
unit_texts = dict(zip(uids, texts))
print(f"유닛 수: {len(uids)}")

questions = load_testset()
tail = load_testset(ROOT / "data" / "testset" / "testset_tail_probe.jsonl")
print(f"평가 질문: {len(questions)}건, 꼬리 프로브: {len(tail)}건")

유닛 수: 494
평가 질문: 557건, 꼬리 프로브: 4건


## 3. 모델 설정

HF 모델카드 기준 실제 사용법 차이:
- **bge-m3 / bge-m3-ko**: `encode(texts)` 그대로, 질의·문서 구분 없음 (`api_style="plain"`)
- **Qwen3-Embedding-8B**: 문서는 그대로, **질의만** `encode([q], prompt_name="query")` (`api_style="prompt_name"`)
- **Nemotron-3-Embed-8B-BF16**: 전용 메서드 `encode_document()` / `encode_query()`가 자동으로
  `"passage: "`/`"query: "` 접두어를 붙임 (`api_style="query_document_methods"`)

8B 모델 2개는 임베딩 차원이 최대 4096(Qwen3는 MRL로 축소 가능)이라 bge류(1024차원)와 벡터 자체는 비교 불가 —
여기선 각 모델 내부에서 독립적으로 랭킹만 매기고, 랭킹 기반 지표(Recall/MRR/AnswerRecall)로만 비교한다.

In [6]:
MODEL_CONFIGS = {
    "bge-m3": {
        "hf_id": "BAAI/bge-m3",
        "api_style": "plain",
        "batch_size": 8,
    },
    "bge-m3-ko": {
        "hf_id": "dragonkue/BGE-m3-ko",
        "api_style": "plain",
        "batch_size": 8,
    },
    "qwen3-embedding-8b": {
        "hf_id": "Qwen/Qwen3-Embedding-8B",
        "api_style": "prompt_name",
        "batch_size": 2,
        "model_kwargs": {"torch_dtype": torch.bfloat16},
        "tokenizer_kwargs": {"padding_side": "left"},
    },
    "nemotron-3-embed-8b": {
        "hf_id": "nvidia/Nemotron-3-Embed-8B-BF16",
        "api_style": "query_document_methods",
        "batch_size": 2,
        "trust_remote_code": True,
    },
}

## 4. 범용 Dense 검색기
`src/retrieval.py`의 `DenseRetriever`와 같은 인터페이스(`search(query, k) -> [(unit_id, score)]`)를 따르되,
모델별 API 차이를 `api_style`로 분기한다. 캐시 키에 **모델 id도 포함**해서(기존 `retrieval.py`는 텍스트 해시만 써서
모델을 바꿔도 캐시가 잘못 재사용될 수 있는 문제가 있었음) 다른 모델끼리 캐시가 섞이지 않게 한다.

In [7]:
CACHE_DIR = (
    Path("/content/drive/MyDrive/kdic_embed_cache")
    if IN_COLAB and Path("/content/drive").exists()
    else REPO_ROOT / "data" / "dense_cache_multi"
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("캐시 위치:", CACHE_DIR)


class GenericDenseRetriever:
    def __init__(self, unit_ids, texts, model_key, cfg):
        self.unit_ids = unit_ids
        self.cfg = cfg
        self.model_key = model_key
        print(f"[{model_key}] 모델 로딩: {cfg['hf_id']}")
        self.model = SentenceTransformer(
            cfg["hf_id"],
            device="cuda",
            trust_remote_code=cfg.get("trust_remote_code", False),
            model_kwargs=cfg.get("model_kwargs", {}),
            tokenizer_kwargs=cfg.get("tokenizer_kwargs", {}),
        )
        cache_path = self._cache_path(texts)
        if cache_path.exists():
            print(f"[{model_key}] 캐시 로드: {cache_path.name}")
            self.doc_emb = np.load(cache_path)
        else:
            print(f"[{model_key}] 인코딩 중... (유닛 {len(texts)}개)")
            self.doc_emb = self._encode_docs(texts)
            np.save(cache_path, self.doc_emb)
            print(f"[{model_key}] 캐시 저장: {cache_path.name}")

    def _cache_path(self, texts):
        # 텍스트 + 모델 id를 같이 해시 — 모델이 달라지면 반드시 다른 캐시 파일이 되도록
        h = hashlib.sha256(
            (self.cfg["hf_id"] + "\x00" + "\x00".join(texts)).encode()
        ).hexdigest()[:16]
        return CACHE_DIR / f"{self.model_key}__{h}.npy"

    def _encode_docs(self, texts):
        bs = self.cfg.get("batch_size", 4)
        if self.cfg["api_style"] == "query_document_methods":
            emb = self.model.encode_document(
                texts, normalize_embeddings=True, batch_size=bs, show_progress_bar=True
            )
        else:
            emb = self.model.encode(
                texts, normalize_embeddings=True, batch_size=bs, show_progress_bar=True
            )
        return np.asarray(emb)

    def _encode_query(self, query):
        style = self.cfg["api_style"]
        if style == "query_document_methods":
            v = self.model.encode_query([query], normalize_embeddings=True)[0]
        elif style == "prompt_name":
            v = self.model.encode([query], prompt_name="query", normalize_embeddings=True)[0]
        else:
            v = self.model.encode([query], normalize_embeddings=True)[0]
        return np.asarray(v)

    def search(self, query, k):
        q = self._encode_query(query)
        scores = self.doc_emb @ q
        ranked = sorted(zip(self.unit_ids, scores.tolist()), key=lambda x: x[1], reverse=True)
        return ranked[:k]

    def unload(self):
        del self.model
        gc.collect()
        torch.cuda.empty_cache()

캐시 위치: /content/drive/MyDrive/kdic_embed_cache


## 5. 모델별 순차 평가
8B 모델 2개를 GPU에 동시에 올리면 메모리가 부족할 수 있으므로, 모델 하나 끝날 때마다 반드시 언로드 후 다음 모델로 넘어간다.

In [8]:
results = {}

for model_key, cfg in MODEL_CONFIGS.items():
    print(f"\n===== {model_key} =====")
    retriever = GenericDenseRetriever(uids, texts, model_key, cfg)
    page_ranked = PageRanked(retriever, unit2page)

    doc_metrics = evaluate(page_ranked, questions)          # Recall@k, MRR (페이지 단위)
    per_type = by_type_mrr(page_ranked, questions)          # 유형별 MRR
    arec = answer_recall(retriever, unit_texts, questions)  # AnswerRecall@5 (전체)
    arec_tail = answer_recall(retriever, unit_texts, tail)  # AnswerRecall@5 (꼬리 프로브)

    results[model_key] = {
        **doc_metrics,
        "AnswerRecall@5": arec,
        "AnswerRecall@5_tail": arec_tail,
        "per_type_mrr": per_type,
    }
    print(model_key, {k: v for k, v in results[model_key].items() if k != "per_type_mrr"})

    retriever.unload()
    del retriever, page_ranked
    gc.collect()
    torch.cuda.empty_cache()


===== bge-m3 =====
[bge-m3] 모델 로딩: BAAI/bge-m3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

[nemotron-3-embed-8b] 인코딩 중... (유닛 494개)


Batches:   0%|          | 0/247 [00:00<?, ?it/s]

[nemotron-3-embed-8b] 캐시 저장: nemotron-3-embed-8b__8dc4f496f13fb929.npy
nemotron-3-embed-8b {'Recall@1': 0.7522441651705566, 'Recall@3': 0.9299820466786356, 'Recall@5': 0.9694793536804309, 'Recall@10': 0.9874326750448833, 'MRR': 0.8458728733863379, 'AnswerRecall@5': 0.9138240574506283, 'AnswerRecall@5_tail': 1.0}


## 6. 비교표

In [9]:
main_cols = [k for k in next(iter(results.values())) if k != "per_type_mrr"]
main_df = pd.DataFrame({m: {c: r[c] for c in main_cols} for m, r in results.items()}).T
main_df

,Recall@1,Recall@3,Recall@5,Recall@10,MRR,AnswerRecall@5,AnswerRecall@5_tail
bge-m3,0.588869,0.838420,0.913824,0.969479,0.723890,0.827648,1.0
bge-m3-ko,0.640934,0.883303,0.951526,0.976661,0.770051,0.879713,1.0
qwen3-embedding-8b,0.639138,0.874327,0.935368,0.980251,0.765021,0.879713,1.0
nemotron-3-embed-8b,0.752244,0.929982,0.969479,0.987433,0.845873,0.913824,1.0


In [10]:
per_type_df = pd.DataFrame({m: r["per_type_mrr"] for m, r in results.items()}).T
per_type_df

,fact,file_download,link_guide,table_lookup,faq
bge-m3,0.678411,0.830882,0.518050,0.833869,0.928205
bge-m3-ko,0.728815,1.000000,0.592149,0.893125,0.886813
qwen3-embedding-8b,0.739154,0.918301,0.506221,0.849375,0.910897
nemotron-3-embed-8b,0.827424,1.000000,0.634685,0.921875,0.933846


## 7. 결과 저장
`docs/retrieval_experiment_results.md`에 이어붙이거나 참조할 수 있도록 JSON으로 저장.

In [11]:
out_path = REPO_ROOT / "docs" / "embedding_model_comparison.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("저장:", out_path)

저장: /content/kdic-crawler/docs/embedding_model_comparison.json


In [15]:
import json
print(json.dumps(results, ensure_ascii=False))

{"bge-m3": {"Recall@1": 0.5888689407540395, "Recall@3": 0.8384201077199281, "Recall@5": 0.9138240574506283, "Recall@10": 0.9694793536804309, "MRR": 0.723890028782308, "AnswerRecall@5": 0.8276481149012568, "AnswerRecall@5_tail": 1.0, "per_type_mrr": {"fact": 0.6784107032012062, "file_download": 0.8308823529411765, "link_guide": 0.518050193050193, "table_lookup": 0.8338690476190477, "faq": 0.9282051282051282}}, "bge-m3-ko": {"Recall@1": 0.6409335727109515, "Recall@3": 0.8833034111310593, "Recall@5": 0.9515260323159784, "Recall@10": 0.9766606822262118, "MRR": 0.7700507252571879, "AnswerRecall@5": 0.8797127468581688, "AnswerRecall@5_tail": 1.0, "per_type_mrr": {"fact": 0.7288152877538355, "file_download": 1.0, "link_guide": 0.5921492921492921, "table_lookup": 0.8931250000000002, "faq": 0.8868131868131868}}, "qwen3-embedding-8b": {"Recall@1": 0.6391382405745063, "Recall@3": 0.874326750448833, "Recall@5": 0.9353680430879713, "Recall@10": 0.9802513464991023, "MRR": 0.7650209455415913, "Answer

## 참고 / 주의사항

- **청킹 모드 고정**: 모델 비교가 목적이라 `MODE="all"`로 고정했다. 청킹×모델 조합까지 보려면 바깥에 `for MODE in [...]` 루프를
  하나 더 감싸면 되는데, 8B 모델 2개가 있어 실행 시간이 곱절로 늘어난다는 점을 감안할 것.
- **차원 비교 불가**: bge류는 1024차원, Qwen3/Nemotron은 최대 4096차원 — 모델 간 벡터를 직접 비교하는 게 아니라
  각 모델이 내놓은 랭킹으로 Recall/MRR/AnswerRecall만 비교하므로 문제없음.
- **GPU 메모리**: 8B 모델은 bf16 기준 가중치만 약 16GB. Colab Pro라도 T4(16GB)는 빠듯할 수 있어 A100/L4 런타임을 권장.
  OOM 나면 `batch_size`를 1로 낮추거나, 8B 모델 2개는 세션을 나눠 따로 실행.
- **Nemotron 사용법은 모델카드 기준 요약**이라, 실제 실행 시 `encode_document`/`encode_query` 메서드가 설치된
  `sentence-transformers` 버전에 없다고 에러나면 최신 버전으로 업그레이드하거나 모델카드의 최신 예제를 다시 확인할 것.
- **세션 재시작 시**: `CACHE_DIR`가 Drive를 가리키고 있으면 이전에 인코딩한 모델은 재다운로드/재인코딩 없이 캐시 재사용됨.